In [2]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
import gradio as gr

In [3]:
def build_model(num_classes=22, Weight_name='',freeze_layers=-50):
    pretrained_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights=None,
        input_shape=(224, 224, 3),
        classes = num_classes
    )

    pretrained_model.trainable = True

    # Freeze layer awal
    for layer in pretrained_model.layers[:freeze_layers]:
        layer.trainable = False

    model = models.Sequential([
        pretrained_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.load_weights(Weight_name)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    print('LOAD SUCESSFULL!!')
    return model

def get_label(idx):
    label = ['Cashew anthracnose', 'Cashew gumosis', 'Cashew healthy', 'Cashew leaf miner', 'Cashew red rust', 'Cassava bacterial blight', 'Cassava brown spot', 'Cassava green mite', 'Cassava healthy', 'Cassava mosaic', 'Maize fall armyworm', 'Maize grasshoper', 'Maize healthy', 'Maize leaf beetle', 'Maize leaf blight', 'Maize leaf spot', 'Maize streak virus', 'Tomato healthy', 'Tomato leaf blight', 'Tomato leaf curl', 'Tomato septoria leaf spot', 'Tomato verticulium wilt']
    return label[idx]


def get_description(label):
    deskripsi_crop = {
        'Cashew anthracnose': 'Penyakit jamur Colletotrichum gloeosporioides yang menyerang daun muda, bunga, dan buah jambu mete, menyebabkan bintik hitam.',
        'Cashew gumosis': 'Penyakit fisiologis atau jamur yang menyebabkan keluarnya cairan getah (gum) berlebih dari batang atau cabang pohon.',
        'Cashew healthy': 'Tanaman jambu mete dalam kondisi sehat, daun hijau segar, tidak ada bercak atau hama.',
        'Cashew leaf miner': 'Hama serangga (ulat pengorok) yang membuat terowongan di dalam jaringan daun, meninggalkan jejak putih berliku.',
        'Cashew red rust': 'Penyakit yang disebabkan oleh alga Cephaleuros virescens, muncul sebagai bercak merah berkarat pada daun.',
        'Cassava bacterial blight': 'Penyakit hawar bakteri (Xanthomonas) yang menyebabkan bercak menyudut pada daun, layu, dan kematian pucuk.',
        'Cassava brown spot': 'Penyakit jamur Cercospora yang menimbulkan bercak cokelat pada daun tua, menyebabkan daun rontok lebih awal.',
        'Cassava green mite': 'Hama tungau hijau yang menghisap cairan daun singkong dari permukaan bawah, menyebabkan bintik klorosis.',
        'Cassava healthy': 'Tanaman singkong sehat dengan pertumbuhan daun optimal dan bebas patogen.',
        'Cassava mosaic': 'Penyakit virus yang ditularkan kutu kebul, menyebabkan daun keriting, belang-belang kuning (mosaik), dan kerdil.',
        'Maize fall armyworm': 'Hama ulat grayak (Spodoptera frugiperda) yang sangat invasif, memakan daun pupus dan tongkol jagung.',
        'Maize grasshoper': 'Hama belalang yang memakan dedaunan tanaman jagung secara langsung.',
        'Maize healthy': 'Tanaman jagung tumbuh subur, daun hijau utuh, dan tongkol berkembang baik.',
        'Maize leaf beetle': 'Kumbang pemakan daun yang menggerogoti jaringan daun hingga berlubang.',
        'Maize leaf blight': 'Penyakit hawar daun (jamur) yang menyebabkan lesi panjang berbentuk cerutu berwarna abu-abu atau cokelat.',
        'Maize leaf spot': 'Penyakit bercak daun (Curvularia/Cercospora) dengan bintik-bintik kecil berwarna cokelat atau kuning.',
        'Maize streak virus': 'Penyakit virus yang menyebabkan garis-garis kuning memanjang sejajar tulang daun.',
        'Tomato healthy': 'Tanaman tomat sehat, daun hijau, batang kokoh, dan bebas dari layu atau bercak.',
        'Tomato leaf blight': 'Penyakit hawar daun (Phytophthora) yang menyebabkan bercak basah kehitaman pada daun dan buah.',
        'Tomato leaf curl': 'Penyakit virus kuning keriting yang menyebabkan daun menguning, melengkung ke atas, dan tanaman kerdil.',
        'Tomato septoria leaf spot': 'Penyakit jamur Septoria yang menimbulkan bercak bulat kecil dengan pinggiran gelap dan pusat berwarna abu-abu.',
        'Tomato verticulium wilt': 'Penyakit layu akibat jamur tanah Verticillium yang menyerang sistem vaskular, menyebabkan tanaman layu perlahan.'
    } 
    return deskripsi_crop[label]

def get_dangerLev(label):
    tingkat_bahaya_crop = {
        'Cashew anthracnose': 'Tinggi',
        'Cashew gumosis': 'Sedang',
        'Cashew healthy': 'Aman',
        'Cashew leaf miner': 'Sedang',
        'Cashew red rust': 'Rendah',
        'Cassava bacterial blight': 'Tinggi',
        'Cassava brown spot': 'Sedang',
        'Cassava green mite': 'Sedang',
        'Cassava healthy': 'Aman',
        'Cassava mosaic': 'Kritis',
        'Maize fall armyworm': 'Kritis',
        'Maize grasshoper': 'Rendah',
        'Maize healthy': 'Aman',
        'Maize leaf beetle': 'Rendah',
        'Maize leaf blight': 'Tinggi',
        'Maize leaf spot': 'Sedang',
        'Maize streak virus': 'Tinggi',
        'Tomato healthy': 'Aman',
        'Tomato leaf blight': 'Tinggi',
        'Tomato leaf curl': 'Kritis',
        'Tomato septoria leaf spot': 'Sedang',
        'Tomato verticulium wilt': 'Tinggi'
    }  
    return tingkat_bahaya_crop[label]

def get_ecoLoss(label):
    kerugian_crop = {
        'Cashew anthracnose': [
            "Menyebabkan keguguran bunga, tunas muda, dan buah mentah (abortus).",
            "Penurunan signifikan pada potensi hasil panen biji mete.",
            "Buah yang terinfeksi memiliki bercak hitam, mengurangi kualitas dan nilai jual."
        ],
        'Cashew gumosis': [
            "Kerusakan jaringan dan pembuluh kayu pada batang dan cabang.",
            "Menghambat transportasi air dan nutrisi (translokasi) ke seluruh bagian pohon.",
            "Jika parah, dapat menyebabkan kematian cabang atau seluruh pohon."
        ],
        'Cashew healthy': [
            "Tidak ada kerugian, potensi hasil panen optimal dan kualitas biji maksimal."
        ],
        'Cashew leaf miner': [
            "Mengganggu proses fotosintesis karena kerusakan luas permukaan daun oleh terowongan larva.",
            "Pertumbuhan tanaman terhambat, terutama pada tanaman muda.",
            "Daun yang rusak rentan terhadap infeksi sekunder patogen lain."
        ],
        'Cashew red rust': [
            "Mengurangi vigor tanaman dan efisiensi fotosintesis.",
            "Defoliasi (rontoknya daun) secara prematur, terutama pada serangan berat."
        ],
        'Cassava bacterial blight': [
            "Penurunan hasil umbi secara drastis (dapat mencapai 100% pada kasus berat).",
            "Menyebabkan layu mendadak dan kematian pucuk (dieback).",
            "Kehilangan bahan tanam karena stek batang tidak bisa digunakan lagi."
        ],
        'Cassava brown spot': [
            "Menyebabkan bercak cokelat dan rontoknya daun tua secara dini.",
            "Mengurangi periode pengisian umbi, berdampak pada penyusutan bobot umbi singkong."
        ],
        'Cassava green mite': [
            "Pertumbuhan tanaman kerdil dengan ruas batang yang memendek.",
            "Menyebabkan bintik-bintik klorosis dan perubahan bentuk daun (deformasi).",
            "Menurunkan bobot dan kualitas umbi singkong."
        ],
        'Cassava healthy': [
            "Tidak ada kerugian, pertumbuhan daun dan umbi mencapai potensi terbaik."
        ],
        'Cassava mosaic': [
            "Gagalnya pembentukan umbi atau umbi yang dihasilkan sangat kecil.",
            "Menyebabkan daun keriting dan gejala mosaik (belang kuning) yang parah.",
            "Kerugian ekonomi total karena tanaman tidak produktif."
        ],
        'Maize fall armyworm': [
            "Kerusakan parah pada titik tumbuh (pupus) tanaman jagung muda.",
            "Kerusakan pada daun dan tongkol yang sedang berkembang, mengakibatkan kegagalan panen.",
            "Salah satu hama yang paling destruktif dan menyebabkan kerugian finansial besar."
        ],
        'Maize grasshoper': [
            "Defoliasi (penggundulan) daun tanaman, mengurangi area fotosintesis.",
            "Menurunkan biomassa tanaman dan vigor, terutama jika serangan terjadi pada fase vegetatif awal."
        ],
        'Maize healthy': [
            "Tidak ada kerugian, tanaman mampu menghasilkan tongkol jagung dengan kualitas dan kuantitas maksimal."
        ],
        'Maize leaf beetle': [
            "Menggerogoti jaringan daun hingga berlubang-lubang ('shot-hole').",
            "Mengurangi efisiensi fotosintesis, berdampak pada pengisian biji."
        ],
        'Maize leaf blight': [
            "Kematian jaringan daun yang luas karena lesi berbentuk cerutu.",
            "Biji jagung yang dihasilkan menjadi kecil, ringan, dan tidak seragam."
        ],
        'Maize leaf spot': [
            "Menyebabkan bercak-bercak kecil yang mengurangi kualitas daun.",
            "Sedikit pengurangan pada berat dan kualitas biji jagung."
        ],
        'Maize streak virus': [
            "Tanaman kerdil dan pertumbuhan terhambat.",
            "Tongkol tidak terisi penuh atau hampa (blanking).",
            "Menyebabkan garis-garis kuning (streak) memanjang sejajar tulang daun, mengurangi fungsi daun."
        ],
        'Tomato healthy': [
            "Tidak ada kerugian, tanaman menghasilkan buah tomat yang berlimpah dan berkualitas baik."
        ],
        'Tomato leaf blight': [
            "Pembusukan daun dan buah secara cepat dalam kondisi kelembaban tinggi.",
            "Menyebabkan kerugian panen total dalam beberapa hari jika tidak dikendalikan.",
            "Buah yang terinfeksi menjadi busuk dan tidak layak jual."
        ],
        'Tomato leaf curl': [
            "Bunga rontok (gagal menjadi buah) jika infeksi terjadi pada fase pembungaan.",
            "Daun menguning, melengkung ke atas, dan tanaman menjadi kerdil.",
            "Kerugian hasil panen sangat tinggi karena tanaman tidak dapat berproduksi."
        ],
        'Tomato septoria leaf spot': [
            "Defoliasi (rontoknya daun) dimulai dari bagian bawah tanaman.",
            "Buah terpapar sinar matahari langsung (sunscald) dan kecil karena minimnya naungan daun."
        ],
        'Tomato verticulium wilt': [
            "Tanaman layu permanen saat fase pengisian buah.",
            "Produksi buah menurun drastis karena gangguan transportasi air dan nutrisi.",
            "Penyakit tular tanah yang sulit dikendalikan setelah menginfeksi."
        ]
    }
    return kerugian_crop[label]

def getHealty(label):
    if (label == 'Tomato healthy') or (label == 'Maize healthy') or (label == 'Cassava healthy') or (label == 'Cashew healthy'):
        return True
    return False

def get_handleStrategy(label):
    penanganan_crop = {
        'Cashew anthracnose': [
            "Lakukan sanitasi dan pangkas bagian tanaman yang terinfeksi parah.",
            "Aplikasi fungisida berbahan aktif tembaga atau azoksistrobin selama periode rentan (pembungaan/buah muda).",
            "Atur jarak tanam untuk meningkatkan sirkulasi udara dan mengurangi kelembaban."
        ],
        'Cashew gumosis': [
            "Perbaiki drainase tanah untuk menghindari genangan air yang memicu gumosis.",
            "Kerok atau pangkas kulit batang yang sakit, lalu oleskan fungisida protektif atau bubur Bordeaux.",
            "Hindari pelukaan mekanis pada batang pohon."
        ],
        'Cashew healthy': [
            "Lakukan pemupukan berimbang (NPK) dan pengairan yang tepat sesuai fase pertumbuhan.",
            "Sanitasi kebun secara rutin dan pemantauan berkala terhadap potensi serangan dini."
        ],
        'Cashew leaf miner': [
            "Manfaatkan musuh alami seperti parasitoid (wasp) sebagai pengendalian hayati.",
            "Gunakan insektisida sistemik jika tingkat serangan tinggi dan mengancam pertumbuhan.",
            "Pangkas dan musnahkan daun yang terinfeksi parah."
        ],
        'Cashew red rust': [
            "Pangkas tajuk pohon yang terlalu rimbun untuk meningkatkan penetrasi cahaya dan udara.",
            "Semprotkan fungisida yang mengandung tembaga jika diperlukan."
        ],
        'Cassava bacterial blight': [
            "Gunakan varietas singkong yang teruji toleran atau tahan penyakit.",
            "Lakukan rotasi tanaman dengan tanaman bukan inang bakteri.",
            "Pastikan hanya menggunakan stek batang yang benar-benar sehat dan bebas patogen."
        ],
        'Cassava brown spot': [
            "Atur jarak tanam agar tidak terlalu rapat untuk mengurangi kelembaban mikro.",
            "Perbaiki kesuburan tanah, terutama dengan peningkatan dosis pupuk Kalium.",
            "Lakukan sanitasi lahan dengan memusnahkan sisa tanaman terinfeksi."
        ],
        'Cassava green mite': [
            "Lepaskan predator alami tungau, seperti Typhlodromalus aripo, sebagai pengendalian hayati.",
            "Aplikasi akarisida jika populasi tungau sudah eksplosif.",
            "Pastikan tanaman mendapatkan air yang cukup karena tungau menyukai kondisi kering."
        ],
        'Cassava healthy': [
            "Lakukan Pemilihan bibit yang bersertifikat dan unggul.",
            "Bersihkan gulma untuk mencegah persaingan nutrisi dan sarang hama."
        ],
        'Cassava mosaic': [
            "Cabut dan bakar segera tanaman yang menunjukkan gejala virus (eradication).",
            "Kendalikan vektor utama, yaitu kutu kebul, menggunakan insektisida atau musuh alami.",
            "Gunakan varietas singkong yang secara genetik resisten terhadap virus."
        ],
        'Maize fall armyworm': [
            "Terapkan Pengendalian Hama Terpadu (PHT) secara menyeluruh.",
            "Gunakan perangkap feromon untuk memantau dan mengurangi populasi ngengat.",
            "Aplikasi insektisida berbahan aktif seperti emamektin benzoat atau spinetoram pada titik tumbuh tanaman."
        ],
        'Maize grasshoper': [
            "Pembersihan gulma di sekitar lahan karena gulma menjadi sarang belalang.",
            "Penggunaan insektisida kontak/lambung jika populasi meledak.",
            "Pengendalian manual (pemungutan) pada serangan skala kecil."
        ],
        'Maize healthy': [
            "Terapkan sistem tanam yang tepat dan pemupukan NPK yang berimbang.",
            "Lakukan penyiangan gulma tepat waktu dan pengairan optimal."
        ],
        'Maize leaf beetle': [
            "Lakukan kutip manual atau pasang perangkap likat berwarna kuning.",
            "Semprotkan insektisida hanya jika ambang kerusakan telah terlampaui."
        ],
        'Maize leaf blight': [
            "Tanam varietas hibrida yang memiliki ketahanan terhadap hawar daun.",
            "Aplikasi fungisida protektif seperti mankozeb atau azoksistrobin pada fase rentan.",
            "Musnahkan sisa-sisa tanaman terinfeksi setelah panen."
        ],
        'Maize leaf spot': [
            "Sanitasi lahan secara menyeluruh, hindari meninggalkan sisa tanaman sakit.",
            "Lakukan rotasi tanaman dengan tanaman yang bukan termasuk famili serealia."
        ],
        'Maize streak virus': [
            "Kendalian populasi wereng daun (Cicadulina mbila) sebagai vektor penular.",
            "Cabut dan musnahkan segera tanaman jagung yang sudah menunjukkan gejala virus."
        ],
        'Tomato healthy': [
            "Penanaman varietas unggul dan pemberian ajir/penopang untuk menopang buah.",
            "Penyiraman teratur pada pagi hari dan pemangkasan tunas air."
        ],
        'Tomato leaf blight': [
            "Hindari metode irigasi curah (sprinkler) yang membuat daun basah, gunakan irigasi tetes.",
            "Lakukan pemotongan daun terinfeksi dan aplikasi fungisida klorotalonil atau mankozeb.",
            "Jaga sirkulasi udara di sekitar tanaman."
        ],
        'Tomato leaf curl': [
            "Kendalikan kutu kebul (Bemisia tabaci) dengan insektisida yang efektif atau mulsa perak.",
            "Cabut dan bakar tanaman yang sudah terinfeksi virus secara total.",
            "Tanam varietas tomat yang resisten."
        ],
        'Tomato septoria leaf spot': [
            "Pastikan daun tidak basah terlalu lama (hindari penyiraman sore hari).",
            "Buang daun paling bawah (basal) yang tua atau terinfeksi.",
            "Lakukan rotasi tanaman dengan tanaman yang bukan inang tomat."
        ],
        'Tomato verticulium wilt': [
            "Lakukan solarisasi tanah sebelum penanaman untuk mengurangi inokulum jamur.",
            "Gunakan varietas tomat yang memiliki kode ketahanan 'V' (Verticillium).",
            "Hindari kelebihan nitrogen, gunakan pemupukan yang berimbang."
        ]
    }
    return penanganan_crop[label]

def list_to_bullets(items):
    return "\n".join([f"- {i}" for i in items])

def predict(image_path,state):
    if image_path is None:
        print("TIDAK ADA GAMBAR")
        return "Tidak ada gambar", "", "", ""

    image = tf.keras.preprocessing.image.load_img(image_path)
    image_array= tf.keras.preprocessing.image.img_to_array(image)
    img_resized = tf.image.resize(image_array, (224, 224)).numpy()
    img_pp = preprocess_input(img_resized)
    pred = model.predict(np.expand_dims(img_pp, axis=0))[0]

    idx = np.argmax(pred)
    label = get_label(idx)

    dampak = list_to_bullets(get_ecoLoss(label))
    penanganan = list_to_bullets(get_handleStrategy(label))
    if(getHealty(label)):
        result = (
        f"Nama : {label}\n"
        f"Deskripsi: {get_description(label)}\n"
        f"Cara menjaga: \n{penanganan}"
    )
    else:
        result = (
            f"Nama : {label}\n"
            f"Deskripsi: {get_description(label)}\n"
            f"Tingkat Bahaya: {get_dangerLev(label)}\n"
            f"Dampak: \n{dampak}\n"
            f"Cara menangani: \n{penanganan}"
        )
    lines = result.strip().split("\n")
    summary = "\n".join(lines[:2]) if len(lines) >= 2 else result
    if image:
        state.append((image,summary))
    return result ,state, state  

In [4]:
model = build_model(Weight_name="efficientnetb0_model_weights.h5")

theme = gr.themes.Soft(primary_hue='emerald',
                       secondary_hue='blue',
                       font= gr.themes.utils.fonts.GoogleFont (name='Source Sans Pro', weights=(400, 600)),
                       font_mono=gr.themes.utils.fonts.LocalFont (name='IBM Plex Mono', weights=(400, 700))
                    )

with gr.Blocks(title="Pest Detector") as demo:

    gr.Markdown("# 🐛 PlantCare")
    gr.Markdown("""
                ## Selamat Datang di Aplikasi PlantCare  
                Aplikasi ini merupakan sistem deteksi hama dan penyakit tanaman berbasis AI. Pengguna dapat mengupload gambar dari tanaman yang terdampak, dan sistem akan bisa mengidentifikasi jenis hama atau penyakit, menampilkan tingkat bahayanya, serta memberikan deskripsi singkat, potensi kerugian, dan rekomendasi penanganan. Aplikasi ini dirancang untuk membantu petani, agronomis, dan praktisi lapangan dalam mengambil keputusan cepat dan akurat untuk menjaga kesehatan tanaman dan meningkatkan hasil panen.

                Tanaman yang bisa diklasifikasi adalah: 
                - Jambu Mede (Cashew) 
                - Singkong (Cassava) 
                - Jagung (Maize) 
                - Tomat (Tomato)

                Total kelas hama (pest) adalah 5:
                - Cashew leaf miner
                - Cassava green mite
                - Maize fall armyworm
                - Maize grasshoper
                - Maize leaf beetle

                Contoh kelas penyakit adalah:
                - Cashew anthracnose
                - Maize leaf blight
                - Tomato septoria leaf spot
                - Dan lain-lain

                Anggota Kelompok:
                - David Goanli - 2702223582
                - Kelson - 2702245135
                - Rafael Komala - 2702227611
                - Meghan Hillary Mardjohan - 2702223443
                - Jason Emanuel Halim - 2702242410
                """)
    with gr.Tabs():

        # ========== TAB 1: Prediksi ==========
        with gr.Tab("Prediksi"):
            with gr.Row():

                # ---- INPUT ----
                with gr.Column():
                    input_path = gr.Image(label="Upload Gambar", type="filepath")
                    predict_btn = gr.Button("Prediksi", variant="primary")
                    reset_btn = gr.ClearButton([input_path])
                    gr.Examples(
                        examples=[
                            ["example/1.jpg"],
                            ["example/2.jpg"],
                            ["example/3.jpg"],
                        ],
                        inputs=[input_path],
                        label="Contoh Gambar"
                    )
                # ---- OUTPUT ----
                with gr.Column():
                    output = gr.Textbox(label="Hasil Analisis Hama", lines=12, interactive=False)

            
            state = gr.State([])

        # ========== TAB 2: Riwayat ==========
        with gr.Tab("Riwayat Prediksi"):
            history = gr.Gallery(label="Riwayat Prediksi", columns=2, object_fit="cover")
    
    predict_btn.click(
                fn=predict,
                inputs=[input_path,state],
                outputs=[output,state, history]
            )
    # Footer
    gr.Markdown("---")
    gr.Markdown("""
    ## **Model Info:** EfficientNetB0
    ## **Cara Pakai:** 
    ## 1. Upload gambar pada kolom yang telah disediakan 
    ## 2. Tekan tombol Prediksi untuk memprediksi gambar yang telah dimasukan
    ## 3. Tunggu sebentar dan hasilnya akan keluar pada kolom sebelah kanan 
    """)

demo.launch(theme=theme)

LOAD SUCESSFULL!!
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
